### Imports

In [1]:
from pathlib import Path
import pandas as pd

### Daten einlesen

In [2]:
df_categories = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/categories.csv")
df_customers = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/customers.csv", dtype={"customer_id": "string"})
df_geolocation = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/geolocation.csv")
df_items = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/items.csv")
df_orders = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/orders.csv", dtype={"order_id": "string", "customer_id": "string"})
df_payments = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/payments.csv")
df_pilot_support = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/pilot_support.csv")
df_products = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/products.csv")
df_reviews = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/reviews.csv")
df_sellers = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/sellers.csv")
df_service_experiment = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/service_experiment.csv")
df_service_observations = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/service_observation.csv")

## 2.1 Daten vorbereiten

### a) Daten Qualität prüfen

1. Was erwarte ich, bzw. Was sagt eine Zeile aus?
2. Woran erkenne ich eine Zeile eindeutig?
3. Wo weichen meine Daten ab?
4. Was kann ich daruas begründet entscheiden?

In [16]:
df_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [17]:
# 1. Eine Zeile soll eine Bestellung darstellen.
print("Bestellungen insgesamt:", len(df_orders))

# 2. Ist die Bestellnummer vollständig und eindeutig?
ids = df_orders["order_id"]
print("Fehlende Bestellnummern:", ids.isna().sum())
print("Mehrfach vorkommende Bestellnummern:", ids[ids.duplicated()].nunique())

# 3. Passen Status und tatsächliches Lieferdatum zusammen?
geliefert = df_orders["order_status"].eq("delivered")
storniert = df_orders["order_status"].eq("canceled")
ist_vorhanden = df_orders["order_delivered_customer_date"].notna()
soll_vorhanden = df_orders["order_estimated_delivery_date"].notna()

print("Geliefert, aber ohne Lieferdatum:", (geliefert & ~ist_vorhanden).sum())
print("Storniert, aber mit Lieferdatum:", (storniert & ist_vorhanden).sum())

# 4. Welche Bestellungen eignen sich für den Vergleich von Ist- und Solltermin?
fuer_liefervergleich = geliefert & ist_vorhanden & soll_vorhanden
print("Geliefert und beide Termine vorhanden:", fuer_liefervergleich.sum())

Bestellungen insgesamt: 99441
Fehlende Bestellnummern: 0
Mehrfach vorkommende Bestellnummern: 0
Geliefert, aber ohne Lieferdatum: 8
Storniert, aber mit Lieferdatum: 6
Geliefert und beide Termine vorhanden: 96470


1. **Was sagt eine Zeile aus?** Eine Zeile beschreibt eine Bestellung. Bei einer zugestellten Bestellung erwarte ich ein tatsächliches Lieferdatum.
2. **Was macht die Zeile eindeutig?** Die `order_id` ist bei allen 99.441 Bestellungen vorhanden und eindeutig.
3. **Wo weichen die Daten ab?** Bei 8 als zugestellt markierten Bestellungen fehlt das Lieferdatum. Bei 6 stornierten Bestellungen ist ein Lieferdatum eingetragen.
4. **Was entscheide ich?** Für den Vergleich von zugesagtem und tatsächlichem Liefertermin verwende ich die 96.470 zugestellten Bestellungen, bei denen beide Termine vorliegen. Für eine Korrektur der auffälligen Einträge fehlt ein Beleg.